# CTM Causal Reasoner for the Switchboard Environment

This notebook implements a Continuous Thought Machine (CTM) to solve tasks in the `Switchboard` environment. The CTM works by iteratively "thinking" about the consequences of its actions before executing a physical toggle. This allows it to reason causally about the environment's rules.

## 1. Setup

First, let's import the necessary libraries and create a `Switchboard` environment. We'll start with a simple scenario.

In [1]:
import torch
import time
import copy
import os

if os.path.basename(os.getcwd()) == 'exercises':
    os.chdir('../')
    
from environments.switchboard.switchboard import Switchboard, RuleBuilder

# Create the environment
env = Switchboard(action_dim=5, obs_dim=5, time_scaling=0.0) # Turn-based for now

# Define a simple scenario: Button 0 toggles Slot 0, Button 1 toggles Slot 1, etc.
for i in range(5):
    env.add_rule(RuleBuilder.toggle(i, i))

print("Switchboard environment created with toggle rules.")

Switchboard environment created with toggle rules.


## 2. The CTM Causal Reasoner Policy

The core of the CTM is a policy that contains an internal "thinking" loop. It simulates the effects of actions on a copy of the environment's state to find an action that achieves a desired goal.

In [2]:
class CtmReasonerPolicy:
    def __init__(self, env, goal_obs):
        self.env = env
        self.goal_obs = goal_obs
        self.action_dim = env.action_dim

    def _evaluate_action(self, candidate_action, current_obs):
        """Helper to simulate an action and return the resulting distance to the goal."""
        mental_env = copy.deepcopy(self.env)
        mental_env.set_obs(current_obs)
        print(mental_env.list_rules())
        
        def mental_policy(obs):
            return candidate_action
        
        obs_list, _ = mental_env.step(mental_policy)
        predicted_obs = obs_list[-1]
        print(predicted_obs)
        
        distance = torch.sum((predicted_obs - self.goal_obs)**2)
        return distance, predicted_obs

    def think(self, current_obs):
        """Internal thinking loop to find the best action by trying single and double presses."""
        best_action = torch.zeros(self.action_dim)
        best_distance = torch.inf
        thought_count = 0

        print(f"--- Starting CTM thought process (trying all single and double actions) ---")
        
        # Phase 1: Try all single-button actions
        for i in range(self.action_dim):
            thought_count += 1
            candidate_action = torch.zeros(self.action_dim)
            candidate_action[i] = 1.0
            
            distance, predicted_obs = self._evaluate_action(candidate_action, current_obs)
            action_indices = torch.where(candidate_action > 0)[0].numpy()
            print(f"Thought {thought_count}: Action {action_indices} -> Predicted Dist: {distance:.4f}")

            if distance < best_distance:
                best_distance = distance
                best_action = candidate_action
                if torch.all(torch.isclose(predicted_obs, self.goal_obs)):
                    print(f"Found optimal action after {thought_count} thoughts.")
                    break # Found a perfect action
        
        # Phase 2: Try all two-button combinations if no perfect action was found
        if not torch.all(torch.isclose(current_obs, self.goal_obs)) and best_distance > 0:
            for i in range(self.action_dim):
                for j in range(i + 1, self.action_dim):
                    thought_count += 1
                    candidate_action = torch.zeros(self.action_dim)
                    candidate_action[i] = 1.0
                    candidate_action[j] = 1.0

                    distance, predicted_obs = self._evaluate_action(candidate_action, current_obs)
                    action_indices = torch.where(candidate_action > 0)[0].numpy()
                    print(f"Thought {thought_count}: Action {action_indices} -> Predicted Dist: {distance:.4f}")

                    if distance < best_distance:
                        best_distance = distance
                        best_action = candidate_action
                        if torch.all(torch.isclose(predicted_obs, self.goal_obs)):
                            print(f"Found optimal action after {thought_count} thoughts.")
                            break # Found a perfect action
                if best_distance == 0:
                  break

        action_indices = torch.where(best_action > 0)[0].numpy()
        print(f"--- Concluded thought process. Best action: {action_indices} ---")
        return best_action

    def __call__(self, observation: torch.Tensor) -> torch.Tensor:
        if torch.all(torch.isclose(observation, self.goal_obs)):
            return torch.zeros(self.action_dim)
        
        return self.think(observation)

def visualize_ctm_policy(policy, initial_obs):
    print("--- Visualizing CTM Policy --- ")
    print(f"Initial Observation: {initial_obs.numpy()}")
    print(f"Goal Observation:    {policy.goal_obs.numpy()}")
    
    # Get the action from the policy by running the 'think' method.
    action = policy.think(initial_obs)
    
    # Simulate the effect of the chosen action.
    final_env = copy.deepcopy(policy.env)
    final_env.set_obs(initial_obs)
    
    def final_policy(obs):
        return action
    
    obs_list, _ = final_env.step(final_policy)
    final_obs = obs_list[-1]
    
    print(f"\nChosen Action: {torch.where(action > 0)[0].numpy()}")
    print(f"Resulting Observation: {final_obs.numpy()}")
    
    if torch.all(torch.isclose(final_obs, policy.goal_obs)):
        print("SUCCESS: The chosen action reaches the goal!")
    else:
        print("FAILURE: The chosen action does not reach the goal.")

## 3. Running the Simulation

Now, let's define a goal state and run the environment with our CTM policy.

In [6]:
# Define a goal observation
goal = torch.zeros(env.obs_dim)
goal[1] = 1.0
goal[3] = 1.0

# Create an instance of our CTM policy
ctm_policy = CtmReasonerPolicy(env, goal_obs=goal)

# Let's visualize the policy's decision for a starting observation
initial_obs = env.reset()
visualize_ctm_policy(ctm_policy, initial_obs)

--- Visualizing CTM Policy --- 
Initial Observation: [0. 0. 0. 0. 0.]
Goal Observation:    [0. 1. 0. 1. 0.]
--- Starting CTM thought process (trying all single and double actions) ---
[{'id': 'toggle_0_to_0', 'description': 'Toggle Button 0 -> Slot 0', 'state': {'is_on': False, 'last_step': -1}}, {'id': 'toggle_1_to_1', 'description': 'Toggle Button 1 -> Slot 1', 'state': {'is_on': False, 'last_step': -1}}, {'id': 'toggle_2_to_2', 'description': 'Toggle Button 2 -> Slot 2', 'state': {'is_on': False, 'last_step': -1}}, {'id': 'toggle_3_to_3', 'description': 'Toggle Button 3 -> Slot 3', 'state': {'is_on': False, 'last_step': -1}}, {'id': 'toggle_4_to_4', 'description': 'Toggle Button 4 -> Slot 4', 'state': {'is_on': False, 'last_step': -1}}]
tensor([1., 0., 0., 0., 0.])
Thought 1: Action [0] -> Predicted Dist: 3.0000
[{'id': 'toggle_0_to_0', 'description': 'Toggle Button 0 -> Slot 0', 'state': {'is_on': False, 'last_step': -1}}, {'id': 'toggle_1_to_1', 'description': 'Toggle Button 1 -> 

## 4. A More Complex Scenario: AND-Gate

Now, let's test the CTM on a more complex problem that requires pressing two buttons simultaneously. We'll create a new environment with an AND-gate rule.

In [3]:
from environments.switchboard.pygame_interface import SwitchboardPygameInterface


print("\n--- Testing with a more complex AND-gate scenario ---")
# Create a new environment for the AND-gate problem
and_env = Switchboard(action_dim=5, obs_dim=5, time_scaling=0.0)

# Rule: Pressing buttons 1 and 2 at the same time 'activates slot 4
and_env.add_rule(RuleBuilder.and_combo([1, 2], 4))

# interface = SwitchboardPygameInterface(
#     and_env,
#     auto_policy=None,  # if you already have a policy you can use this to execute your policy
# )

# # Run the interface
# interface.run()

# Goal: Activate slot 4
and_goal = torch.zeros(and_env.obs_dim)
and_goal[4] = 1.0

# Create a new CTM policy for this environment
and_ctm_policy = CtmReasonerPolicy(and_env, goal_obs=and_goal)

# Visualize the CTM policy solving the AND-gate problem
initial_and_obs = and_env.reset()
visualize_ctm_policy(and_ctm_policy, initial_and_obs)

pygame 2.2.0 (SDL 2.32.54, Python 3.11.14)
Hello from the pygame community. https://www.pygame.org/contribute.html

--- Testing with a more complex AND-gate scenario ---
--- Visualizing CTM Policy --- 
Initial Observation: [0. 0. 0. 0. 0.]
Goal Observation:    [0. 0. 0. 0. 1.]
--- Starting CTM thought process (trying all single and double actions) ---
[{'id': 'and_1,2_to_4', 'description': 'Buttons [1, 2] -> Slot 4 (AND)', 'state': {}}]
tensor([0., 0., 0., 0., 0.])
Thought 1: Action [0] -> Predicted Dist: 1.0000
[{'id': 'and_1,2_to_4', 'description': 'Buttons [1, 2] -> Slot 4 (AND)', 'state': {}}]
tensor([0., 0., 0., 0., 0.])
Thought 2: Action [1] -> Predicted Dist: 1.0000
[{'id': 'and_1,2_to_4', 'description': 'Buttons [1, 2] -> Slot 4 (AND)', 'state': {}}]
tensor([0., 0., 0., 0., 0.])
Thought 3: Action [2] -> Predicted Dist: 1.0000
[{'id': 'and_1,2_to_4', 'description': 'Buttons [1, 2] -> Slot 4 (AND)', 'state': {}}]
tensor([0., 0., 0., 0., 0.])
Thought 4: Action [3] -> Predicted Dis

/home/mwuerstle@corp.exxcellent.de/miniconda3/envs/agi/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
